# SQL for Data Analysts

> **Track:** Data Science / ML Engineer | **Level:** Beginner | **Time:** 2 hours

## Overview
SQL is the most universally required skill in data roles. This notebook covers everything from basic queries to window functions and CTEs — using SQLite (built into Python) so you can run everything locally.

### What You'll Learn
- Database setup and CREATE/INSERT
- SELECT, WHERE, ORDER BY, LIMIT
- GROUP BY, HAVING, aggregate functions
- JOINs: INNER, LEFT, RIGHT
- Subqueries and CTEs
- Window functions: ROW_NUMBER, LAG, LEAD, RANK
- Combining SQL results with pandas

```bash
# sqlite3 and pandas are built-in / standard
pip install pandas
```

In [ ]:
# Setup: create an in-memory SQLite database with sample e-commerce data
import sqlite3
import pandas as pd
import numpy as np
from datetime import date, timedelta

conn = sqlite3.connect(":memory:")
cur = conn.cursor()

# Create tables
cur.executescript("""
    CREATE TABLE customers (
        customer_id INTEGER PRIMARY KEY,
        name TEXT NOT NULL,
        city TEXT,
        signup_date TEXT
    );
    CREATE TABLE products (
        product_id INTEGER PRIMARY KEY,
        name TEXT,
        category TEXT,
        price REAL
    );
    CREATE TABLE orders (
        order_id INTEGER PRIMARY KEY,
        customer_id INTEGER,
        product_id INTEGER,
        quantity INTEGER,
        order_date TEXT,
        FOREIGN KEY(customer_id) REFERENCES customers(customer_id),
        FOREIGN KEY(product_id) REFERENCES products(product_id)
    );
""")

# Insert sample data
customers = [(1,'Alice','New York','2023-01-15'),(2,'Bob','Chicago','2023-02-20'),(3,'Carol','New York','2023-03-10'),(4,'Dave','Los Angeles','2023-04-05'),(5,'Eve','Chicago','2023-05-18')]
products = [(1,'Laptop','Electronics',999.99),(2,'Headphones','Electronics',149.99),(3,'Python Book','Books',39.99),(4,'Desk Chair','Furniture',299.99),(5,'Monitor','Electronics',449.99)]
orders = [(1,1,1,1,'2024-01-10'),(2,1,2,2,'2024-01-15'),(3,2,3,1,'2024-01-20'),(4,3,1,1,'2024-02-01'),(5,3,5,1,'2024-02-15'),(6,1,5,1,'2024-03-01'),(7,4,4,2,'2024-03-10'),(8,5,2,1,'2024-03-20'),(9,2,1,1,'2024-04-01'),(10,1,3,3,'2024-04-15')]

cur.executemany("INSERT INTO customers VALUES (?,?,?,?)", customers)
cur.executemany("INSERT INTO products VALUES (?,?,?,?)", products)
cur.executemany("INSERT INTO orders VALUES (?,?,?,?,?)", orders)
conn.commit()

print("Database created: customers, products, orders tables")
print(f"Rows: {len(customers)} customers, {len(products)} products, {len(orders)} orders")

## 1. Basic Queries: SELECT, WHERE, ORDER BY

In [ ]:
# Basic SELECT queries with filtering and sorting

def q(sql: str) -> pd.DataFrame:
    """Helper: run a SQL query and return a pandas DataFrame."""
    return pd.read_sql_query(sql, conn)

print("=== 1. All customers from New York ===")
print(q("SELECT * FROM customers WHERE city = 'New York'"))

print("\n=== 2. Top 3 most expensive products ===")
print(q("SELECT name, category, price FROM products ORDER BY price DESC LIMIT 3"))

print("\n=== 3. Orders in Q1 2024 ===")
print(q("SELECT order_id, customer_id, order_date FROM orders WHERE order_date BETWEEN '2024-01-01' AND '2024-03-31'"))

print("\n=== 4. Electronics under $500 ===")
print(q("SELECT name, price FROM products WHERE category = 'Electronics' AND price < 500 ORDER BY price"))

## 2. GROUP BY, HAVING & Aggregations

In [ ]:
# Aggregations and GROUP BY

print("=== 1. Total revenue by product category ===")
print(q("""
    SELECT p.category,
           COUNT(o.order_id) AS num_orders,
           SUM(o.quantity * p.price) AS total_revenue,
           AVG(p.price) AS avg_price
    FROM orders o
    JOIN products p ON o.product_id = p.product_id
    GROUP BY p.category
    ORDER BY total_revenue DESC
"""))

print("\n=== 2. Customers with more than 2 orders (HAVING) ===")
print(q("""
    SELECT c.name, COUNT(o.order_id) AS order_count,
           SUM(o.quantity * p.price) AS lifetime_value
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    JOIN products p ON o.product_id = p.product_id
    GROUP BY c.customer_id, c.name
    HAVING order_count > 2
    ORDER BY lifetime_value DESC
"""))

## 3. JOINs and Subqueries

In [ ]:
# JOINs and subqueries

print("=== 1. INNER JOIN: all order details ===")
print(q("""
    SELECT c.name AS customer, p.name AS product,
           o.quantity, ROUND(o.quantity * p.price, 2) AS order_total, o.order_date
    FROM orders o
    INNER JOIN customers c ON o.customer_id = c.customer_id
    INNER JOIN products p ON o.product_id = p.product_id
    ORDER BY o.order_date
""").head(6))

print("\n=== 2. LEFT JOIN: customers who never ordered ===")
print(q("""
    SELECT c.name, c.city, o.order_id
    FROM customers c
    LEFT JOIN orders o ON c.customer_id = o.customer_id
    WHERE o.order_id IS NULL
"""))

print("\n=== 3. Subquery: customers who spent above average ===")
print(q("""
    SELECT c.name, ROUND(SUM(o.quantity * p.price), 2) AS spent
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    JOIN products p ON o.product_id = p.product_id
    GROUP BY c.customer_id, c.name
    HAVING spent > (
        SELECT AVG(sub.total) FROM (
            SELECT SUM(o2.quantity * p2.price) AS total
            FROM orders o2 JOIN products p2 ON o2.product_id = p2.product_id
            GROUP BY o2.customer_id
        ) sub
    )
"""))

## 4. Window Functions and CTEs

In [ ]:
# Window functions and CTEs

print("=== 1. CTE: monthly revenue trend ===")
print(q("""
    WITH monthly_revenue AS (
        SELECT SUBSTR(o.order_date, 1, 7) AS month,
               SUM(o.quantity * p.price) AS revenue
        FROM orders o
        JOIN products p ON o.product_id = p.product_id
        GROUP BY month
    )
    SELECT month, ROUND(revenue, 2) AS revenue,
           ROUND(SUM(revenue) OVER (ORDER BY month), 2) AS cumulative_revenue
    FROM monthly_revenue
    ORDER BY month
"""))

print("\n=== 2. Window: RANK customers by spend ===")
print(q("""
    WITH customer_spend AS (
        SELECT c.name, ROUND(SUM(o.quantity * p.price), 2) AS total_spent
        FROM customers c
        JOIN orders o ON c.customer_id = o.customer_id
        JOIN products p ON o.product_id = p.product_id
        GROUP BY c.customer_id, c.name
    )
    SELECT name, total_spent,
           RANK() OVER (ORDER BY total_spent DESC) AS spend_rank,
           ROUND(total_spent / SUM(total_spent) OVER () * 100, 1) AS pct_of_total
    FROM customer_spend
"""))

## Exercises

1. **Cohort analysis**: Write a SQL query that groups customers by their `signup_date` month (cohort) and calculates the average number of orders per customer in each cohort. Which cohort is most engaged?

2. **Product recommendations**: Write a query that finds products frequently purchased together (in different orders by the same customer). Use a self-join on the orders table grouped by `customer_id` and `product_id` pairs, filtering to pairs appearing at least twice.

3. **Running average with LAG**: Use the `LAG` window function to compute a 3-month rolling average revenue. For each month, calculate `(current_month + prev_month + month_before_prev) / 3` and compare it to the actual monthly revenue to smooth out noise.